# Lesson 5 · An ID card for every frequency — the magnitude and phase of a complex number, the native language of the FFT

**Learning goals**
1. Understand the rectangular coordinates $(a,b)$ and polar coordinates $(r, \theta)$ of a complex number `a+bj` (terms summarized in the table below)
2. Implement `magnitude_phase(z)`, returning the tuple `(magnitude r, phase φ)`
3. Understand why the FFT outputs complex numbers: magnitude = how loud that frequency is, phase = how it lines up in time
4. Lay the groundwork for Euler's formula `e^{jθ} = cos(θ) + j·sin(θ)` in L06

**Carried over from L04**: the amplitude A and phase φ of a `sinusoid`
correspond, in the FFT output, to the **magnitude** and **phase** of a complex number.
In this lesson you read those two quantities out of a complex number yourself.

**Aurora connection**: `magnitude_spectrogram` = take the magnitude of the complex number in every FFT frequency bin;
phase becomes crucial later in **waveform reconstruction** (L45) — without phase you can usually recover "how loud", but not "how the waveform lines up".

← **Previous lesson** · [L04 · The three ingredients of a sine wave](L04_trig.en.ipynb)

> Last lesson covered **the three ingredients of a sine wave**: frequency sets pitch, amplitude sets loudness, phase sets the starting point — and implemented them by hand.
> This lesson explores **the magnitude and phase of a complex number**: an ID card for every frequency, the native language of the FFT.

## This lesson's story: issuing an "ID card" to every frequency component

When the FFT analyzes a sound, every frequency component gets a complex number `X[k] = a + bj`.

But what *is* a "complex number"? It isn't make-believe — it's a **two-dimensional ID card** that records two things at once:
- **how loud this frequency is** (amplitude / magnitude)
- **where this frequency starts off** (phase)

One ID card, two pieces of information, packed together.

Picture drawing an arrow on a blackboard. The arrow's **length** is the amplitude; its **direction** is the phase. That arrow is the complex number: in polar coordinates it's **$(r, \theta)$** (length + direction), in rectangular coordinates it's `a + bj`.
(Later you'll see the direction written in the notation $r\cdot e^{i\theta}$ — for now just get acquainted; **L06 takes it apart**.)

| Representation | What it records | Formula |
|---|---|---|
| Rectangular (a+bj) | real part a + imaginary part b | a = r·cos(θ), b = r·sin(θ) |
| Polar (r, θ) | magnitude r + phase θ | r = √(a²+b²), θ = arctan2(b, a) |

In this lesson you'll implement `magnitude_phase(z)`, reading both sides of this "ID card" straight off a complex number.

> 📌 **Two new faces — let's say hello first**: the `r·e^{iθ}` notation in the table only needs a nodding acquaintance this lesson — read it as shorthand for **"an arrow of length r pointing in direction θ"**. Why the direction is written as "e to the iθ" is the main topic of the next lesson, L06 (Section 1 gives you a "black-box user's manual" for it). And `arctan2`, the two-argument arctangent, isn't something you missed in high school either — right before the hands-on part of Section 2 there's a short section that explains it thoroughly.

## 🤔 Think first: without complex numbers, how many numbers does it take to describe one frequency?

**Don't look at the formulas yet — think for 30 seconds.** Suppose you could only use real numbers. To fully describe "one frequency component", how many numbers would you need to write down, at minimum?

- Just an **amplitude**? — Not enough. Two equally loud tones can have different "start times".
- So add a **phase**. That makes at least **two numbers**: one for how loud, one for where it starts.
- An equivalent way to write it: one coefficient in front of `cos` + one coefficient in front of `sin` — still **two degrees of freedom**.

A frequency is born with **two degrees of freedom**. And a complex number `a + bj` holds exactly two numbers — one half for magnitude (amplitude), the other for phase. A perfect fit.

> So complex numbers aren't mathematical showing off; they're an **engineering language**: they pack "the two numbers you need to describe a frequency" into a single object you can multiply and rotate directly. That's exactly why the FFT speaks complex numbers as its native language.

## ⚠️ Common misconceptions

> Don't think of complex numbers as "made-up / nonexistent numbers". A complex number is really **a package of two real degrees of freedom, "magnitude + phase"** — the length and direction of an arrow, both visible and measurable. The word "imaginary" is just a bad name left over from history.

## 1. Basic operations on complex numbers

Ever wondered why mathematicians invented an "imaginary unit" i (j in Python)?

It turns out the real number line has only one dimension. Multiplying by -1 "flips the direction", but flipping isn't enough — some rotations need to **turn 90°**, not 180°. The defining idea of i is "multiplying by i = rotating 90° counterclockwise":

```
i² = -1 (turning 90° twice = turning 180° = multiplying by -1)
```

So a complex number `a + bj` is not two unrelated numbers — it's **a point on a two-dimensional plane**:
- real part a → horizontal direction
- imaginary part b → vertical direction

Python supports complex numbers natively; NumPy's `np.exp(1j * theta)` directly gives the point on the unit circle at angle θ.

Hold on — "multiplying by i = rotating 90° counterclockwise", says who? Announcing it doesn't count. Below we first derive it in two lines using the distributive law, then use code to measure the coordinates, magnitude, and phase before and after the rotation. (As for the `np.exp(1j * theta)` in the previous sentence, that's not something you "should already know" either — later in this section there's a dedicated "black-box user's manual" for it.)

### 🔍 A two-line derivation: why is "multiplying by i" a 90° counterclockwise turn?

The definition you learned in high school is `i² = -1`, yet this lesson takes "rotating 90°" as the starting point — these are really two sides of the same coin. Below we derive it with no leap of faith, using only the **distributive law** (the one you've known since middle school).

Take any complex number `a + bj`, multiply by i, and expand term by term:

```
(a + bj) · i = a·i + b·i² = a·i + b·(-1) = -b + a·i
```

In other words: **the point (a, b) gets moved to the point (-b, a).** This move — "swap the coordinates and put a minus sign in front" — is exactly a 90° counterclockwise turn. Why can we say that? Three pieces of evidence you can check one by one:

1. **The length doesn't change**: the original magnitude is √(a²+b²), the new one is √((-b)²+a²) = √(a²+b²), identical — so it can only be a rotation or a reflection, never a stretch;
2. **Try the simplest arrow**: the arrow (1, 0) on the x-axis becomes (0, 1), going from "pointing right" to "pointing up" — a 90° counterclockwise turn, sure enough;
3. **Try concrete numbers**: (3, 4) becomes (-4, 3) — plot these two points on scratch paper and the arrow has turned exactly one right angle counterclockwise (the code cell below measures it for you).

And turning once more? By the same rule, (-b, a) becomes (-a, -b), the point directly opposite (a, b) — two 90° turns = a 180° turn = multiplying by -1. **That is the geometric origin of `i² = -1`**: it isn't an arbitrary quirk someone decreed, but plain common sense: "two right-angle turns make a U-turn".

> A signpost while we're here: later this lesson leans repeatedly on a more general fact — **multiplying by a complex number with "magnitude 1 and phase φ" = rotating the arrow counterclockwise by φ** (multiplying by i is just the special case φ=90°). Its two-line proof has to wait for Euler's formula in L06; for now the code below "measures" it so you can see it for yourself.

In [1]:
# Aurora matplotlib bootstrap
from pathlib import Path
import sys

_root = None
_cwd = Path.cwd().resolve()
for _candidate in (_cwd, *_cwd.parents):
    if (_candidate / '_matplotlib_bootstrap.py').exists():
        _root = _candidate
        break
if _root is None:
    _notebooks_dir = _cwd / 'notebooks'
    if _notebooks_dir.exists():
        for _found in _notebooks_dir.rglob('_matplotlib_bootstrap.py'):
            _root = _found.parent
            break
if _root is not None and str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from _matplotlib_bootstrap import apply as _aurora_mpl_apply
_aurora_mpl_apply()


In [ ]:
import numpy as np

# See for yourself: multiply by i — does (3,4) really become (-4,3)?
z = 3 + 4j
w = z * 1j
print(f'before rotation z   = {z}    coords ({z.real:.0f}, {z.imag:.0f}), magnitude {abs(z):.4f}, phase {np.angle(z):.4f}')
print(f'after rotation  z*i = {w}   coords ({w.real:.0f}, {w.imag:.0f}), magnitude {abs(w):.4f}, phase {np.angle(w):.4f}')
print(f'phase difference = {np.angle(w) - np.angle(z):.4f}, and π/2 = {np.pi/2:.4f} — exactly a 90° counterclockwise turn, magnitude untouched')
assert np.isclose(abs(w), abs(z)) and np.isclose(np.angle(w) - np.angle(z), np.pi/2)

print()
# Now the general case: "multiplying by i" isn't special — multiplying by any "magnitude-1" complex number is a pure rotation.
# Build a "magnitude 1, direction 30°" complex number with cos+jsin (this notation is formally introduced in the next section):
rot30 = np.cos(np.pi/6) + 1j * np.sin(np.pi/6)
w30 = z * rot30
print(f'z times "the unit complex number pointing at 30°": magnitude {abs(w30):.4f} (unchanged), '
      f'phase {np.angle(w30):.4f} = original phase {np.angle(z):.4f} + π/6 ({np.pi/6:.4f})')
assert np.isclose(abs(w30), abs(z)) and np.isclose(np.angle(w30), np.angle(z) + np.pi/6)
print('✅ Multiplying by a unit complex number = turn the direction, keep the length; "multiplying by i" is just the special case where the angle is exactly 90°.')

### 📦 Black-box user's manual: `np.exp(1j * theta)` (used as a shortcut this lesson; L06 takes it apart)

First, something you may already be muttering about: in high school e≈2.718 and `e^x` is exponential growth that "keeps getting bigger" — so what on earth is **e to an imaginary power**? Why doesn't it grow, and instead sit politely on the unit circle?

**That question is completely fair, and this lesson doesn't intend to answer it** — that's the main topic of the next lesson, L06 on Euler's formula. This lesson just signs a "black-box contract" with you:

> **Contract**: `np.exp(1j * theta)` is a shortcut. Feed it an angle θ and it hands back **the point on the unit circle at angle θ** — **exactly the same** as `np.cos(theta) + 1j*np.sin(theta)`. Anywhere in this lesson you see `np.exp(1j*θ)`, you can mentally replace it with `cos(θ) + sin(θ)j` — and your high-school math is entirely enough for that.

Why meet it now? Because everyone in the FFT world writes `e^{jθ}` (the `r·e^{iθ}` in the table in this lesson's story cell is the same notation: **an arrow of length r pointing in direction θ**). Get acquainted now, and when L06 reveals how it works you'll get that satisfying "oh, so *that's* what it is" moment.

The check cell below verifies this contract in code: it compares the two forms point by point, to see whether they're really equal everywhere and whether the magnitude really is always 1. **Once you've seen it with your own eyes, you can safely use it as a tool.**

In [ ]:
import numpy as np

# Contract check: np.exp(1j*θ) and np.cos(θ) + 1j*np.sin(θ) are equal everywhere
for theta in [0.0, np.pi/4, np.pi/2, np.pi, 5.0]:
    z_exp = np.exp(1j * theta)                 # black-box form (this lesson's shortcut)
    z_cos = np.cos(theta) + 1j * np.sin(theta) # form you can read with high-school math
    assert np.isclose(z_exp, z_cos), 'the two forms must be equal'
    print(f'θ={theta:6.3f}  np.exp(1j*θ)=({z_exp.real:+.4f}, {z_exp.imag:+.4f})   '
          f'cosθ+j·sinθ=({z_cos.real:+.4f}, {z_cos.imag:+.4f})   |z|={abs(z_exp):.4f}')

print()
print('✅ The two forms are identical: whenever np.exp(1j*θ) appears later in this lesson,')
print('   you can mentally swap it for np.cos(θ) + 1j*np.sin(θ).')
print('   Note the rightmost column: the magnitude is always 1 — it never "keeps getting bigger", it just circles the unit circle.')
print('   Why would an exponential politely go in circles? That is exactly the question Euler\'s formula answers in L06.')

## Into the lab: angles, coordinates, and rotation

The unit complex number for angle θ is `cos(θ) + sin(θ)j`: the real part is the horizontal component, the imaginary part the vertical component, and the magnitude is always 1. Below we go once around the circle to check how polar and rectangular coordinates convert into each other.

In [ ]:
import numpy as np
z = 3 + 4j
print('real part:', z.real, ' imaginary part:', z.imag)
print('magnitude |z| =', abs(z), ' (=√(3²+4²)=5)')
print('phase (radians) =', np.angle(z))

## Hands-on: a complex number is "a position under rotation"

Look at the `real` and `imag` columns below: every time the angle grows by π/2, the point moves one step counterclockwise around the unit circle. The `magnitude` column stays at 1; only the `phase` changes.

In [ ]:
import numpy as np

angles = np.array([0, np.pi/2, np.pi, 3*np.pi/2])
z = np.exp(1j * angles)

print('angle =', np.round(angles, 3))
print('real part cos =', np.round(z.real, 3))
print('imag part sin =', np.round(z.imag, 3))
print('complex z =', np.round(z, 3))


## Code experiment: one full turn

Take 8 evenly spaced angles from 0 to 2π, print each point's real part, imaginary part, magnitude, and phase, and confirm that on the unit circle the magnitude is always 1 and the phase is just the angle itself.

In [6]:
import numpy as np

for k in range(9):
    theta = 2 * np.pi * k / 8
    z = np.exp(1j * theta)
    print(f'k={k} theta={theta:.2f} -> ({z.real:+.3f}, {z.imag:+.3f})')


k=0 theta=0.00 -> (+1.000, +0.000)
k=1 theta=0.79 -> (+0.707, +0.707)
k=2 theta=1.57 -> (+0.000, +1.000)
k=3 theta=2.36 -> (-0.707, +0.707)
k=4 theta=3.14 -> (-1.000, +0.000)
k=5 theta=3.93 -> (-0.707, -0.707)
k=6 theta=4.71 -> (-0.000, -1.000)
k=7 theta=5.50 -> (+0.707, -0.707)
k=8 theta=6.28 -> (+1.000, -0.000)


## 🧭 New tool reporting for duty: `arctan2` — the arctangent that knows its quadrants

We're about to compute phases by hand, but there's a new face in the toolbox to introduce first: `np.arctan2`. **It's not something you missed in high school** — high school only teaches the one-argument arctan; the two-argument version was built by engineers to fix a birth defect in arctan.

**What's the defect?** If you try to find the phase by "looking up the angle from the slope", you'd naturally write `arctan(b/a)`. But the instant you compute the division `b/a`, the quadrant information is wiped out:

- The points `3+4j` (upper right) and `-3-4j` (lower left) point northeast and southwest — **exactly opposite** directions — yet `b/a` is 4/3 for both. The division cancels the two minus signs, so arctan can only give one answer;
- Worse, when `a = 0` (arrow pointing straight up/down), `b/a` divides by zero and can't be computed at all.

arctan only eats one number, the "slope", so it can only ever cover the right half of the circle (quadrants I and IV, output range just (−π/2, π/2)).

**arctan2's fix is simple**: stop dividing — hand b and a to the function **separately**. Then the signs of b and a both survive, the function can tell which of the four quadrants the arrow points into, and the output range stretches to a full circle: `(-π, π]`. That's why it insists on two arguments — **the second argument isn't redundant; it's the rescuer of the quadrant information the division would have murdered**.

> ⚠️ **Order warning (the easiest place to trip up)**: it's written `np.arctan2(b, a)` = `arctan2(imag, real)` = `arctan2(y, x)` — **y first, then x**, the opposite of how you normally write coordinates (x, y)! It mimics the fraction b/a: "numerator first, denominator second". Even sneakier: for a test value like `3+4j` where both components are positive, writing it backwards as `arctan2(3, 4)` gives 0.6435 — a perfectly normal-looking number, and only an assertion will expose it. If a phase assertion fails in the exercise, check the argument order first.

The comparison experiment below turns every sentence above into runnable evidence.

In [ ]:
import numpy as np

print('[Quadrant I: 3+4j] slope b/a = 4/3 — the two functions are neck and neck')
print(f'  arctan(4/3)     = {np.arctan(4/3):.4f}  ✅')
print(f'  arctan2(4, 3)   = {np.arctan2(4, 3):.4f}  ✅ matches the line above')
print()
print('[Quadrant III: -3-4j] slope b/a = (-4)/(-3) = 4/3 — exactly the same as above!')
print(f'  arctan(-4/-3)   = {np.arctan(-4/-3):.4f}  ❌ the minus signs cancel in the division, the answer is still quadrant I')
print(f'  arctan2(-4, -3) = {np.arctan2(-4, -3):.4f}  ✅ ≈ -2.214, quadrant III (pointing lower left) — correct')
print()
print('[Quadrant II: -3+4j]')
print(f'  arctan(4/-3)    = {np.arctan(4/-3):.4f}  ❌ gives a quadrant IV angle — wrong')
print(f'  arctan2(4, -3)  = {np.arctan2(4, -3):.4f}  ✅ ≈ +2.214, quadrant II (pointing upper left)')
print()
print('[Real part 0: the pure imaginary 1j]')
print('  arctan(1/0) — division by zero, crashes outright')
print(f'  arctan2(1, 0)   = {np.arctan2(1, 0):.4f}  ✅ = π/2, straight up, no sweat')
print()
print('[Order trap] same 3+4j:')
print(f'  arctan2(4, 3)   = {np.arctan2(4, 3):.4f}  ✅ correct (imaginary first, real second)')
print(f'  arctan2(3, 4)   = {np.arctan2(3, 4):.4f}  ❌ backwards! 0.6435 also looks like a "perfectly reasonable" number — the hardest kind to catch')

## 2. ✏️ Implement `magnitude_phase(z)` returning `(magnitude, phase)`

**Line of reasoning**:
1. The magnitude is the complex number's distance from the origin: `r = √(real² + imag²)`. Python's built-in `abs(z)` returns exactly this, and works on both scalars and numpy arrays.
2. The phase is the angle from the positive x-axis: `θ = arctan2(imag, real)`. `np.angle(z)` wraps this computation; its output range is `(-π, π]`, in radians.
3. We return the tuple `(magnitude, phase)` because FFT post-processing usually needs both at once — more efficient than two separate calls.

**Reference input and output**: `z = 3 + 4j` → magnitude = 5.0, phase ≈ 0.9273 radians (about 53°, i.e. arctan(4/3))

> 💡 Need a hint? After finishing the exercise you can consult the reference implementation in the `solutions/` folder.


### Before writing code, fill in the variable table

Be clear on three things before writing `magnitude_phase`:
- Input: a complex number `z` (Python or numpy complex both work)
- Key steps: `abs(z)` for the magnitude; `np.angle(z)` for the phase, range (−π, π], in radians
- Return: the tuple `(magnitude, phase)`, both floats

### 📐 Rules for the manual implementation (try it yourself first, then look at hints)

**Step 1 (required)**: fill in the stub using only these tools:
- `z.real`, `z.imag` — read the real and imaginary parts
- `np.sqrt()` — square root
- `np.arctan2(b, a)` — arctangent, returns (-π, π]

You are **not allowed** to call `abs(z)` or `np.angle(z)` directly in the stub (that would mean not implementing anything).

Once the assertions pass, add a comment at the end of the function: `# equivalent to: return abs(z), np.angle(z)` — and compare whether the two versions give exactly the same results.

> ⚠️ **One more reminder about argument order**: `np.arctan2(z.imag, z.real)` — **imaginary first, real second** (y before x, the opposite of how coordinates are usually written). Writing it backwards as `arctan2(z.real, z.imag)` gives 0.6435 for `3+4j`, also a "perfectly reasonable" number that only an assertion can catch. When the assertion reports "phase should be 0.9273", suspect number one is the argument order (see the "New tool reporting for duty: arctan2" section above).

In [ ]:
def magnitude_phase(z):
    # ✏️ TODO: implement using only z.real, z.imag, np.sqrt, np.arctan2 (no direct abs/np.angle)
    raise NotImplementedError("Please implement magnitude_phase(z) using z.real, z.imag, np.sqrt, np.arctan2")


In [ ]:
import numpy as np
try:
    mag, ph = magnitude_phase(3 + 4j)
    print('magnitude =', mag, ' phase =', round(ph, 4))
    assert abs(mag - 5.0) < 1e-9, f"magnitude should be 5.0, got {mag}"
    assert abs(ph - np.arctan2(4, 3)) < 1e-9, f"phase should be {np.arctan2(4,3):.4f}, got {ph}"
    print('✅ Passed: you can read magnitude and phase out of a complex number.')
except (NotImplementedError, TypeError) as e:
    print(f'⚠️ Not implemented yet: {e}')
    print('  Go back to the magnitude_phase function above and fill it in with z.real/z.imag/np.sqrt/np.arctan2.')


**🔗 Aurora connection**: `magnitude_spectrogram` takes exactly the magnitude of the FFT's complex output; phase is key in vocoders and reconstruction. Next lesson: tying complex numbers to trigonometry — Euler's formula.

In [ ]:
for k in range(8):
    theta = 2*np.pi*k/8
    z = np.exp(1j*theta)
    radius = abs(z)
    print(f'k={k} | z=({z.real:+.2f}, {z.imag:+.2f}) | radius={radius:.2f}')
print('Every point has radius close to 1, so they all lie on the unit circle.')


## Parameter experiment: magnitude and phase of 8 points on the unit circle

Build `z = np.exp(1j * np.linspace(0, 2*np.pi, 8, endpoint=False))`, print each point's magnitude and phase, and confirm the magnitude is always 1.0 and the phases are evenly spaced in steps of π/4. Note that `np.angle` outputs in the range (−π, π], so angles beyond π (5π/4, 3π/2, 7π/4) wrap around and show up as negative values (−3π/4, −π/2, −π/4).

> ⚠️ **Don't drop `endpoint=False`!** It means "don't include the endpoint 2π". On a circle, 2π and 0 are **the same direction**, so if 2π is included too (the default behavior of `np.linspace`), the first and last of the 8 points coincide — you actually get only 7 distinct directions, and the spacing becomes 2π/7 ≈ 0.898 instead of the π/4 ≈ 0.785 the text promises. If you built it from the formula first and found the step doesn't match — you didn't mistype; you just left out this argument.

Then swap `3 + 4j` for `3 - 4j` and watch the phase go from +0.9273 to -0.9273: negating the imaginary part negates the phase while the magnitude stays the same — evidence that magnitude and phase are independent.

In [ ]:
import numpy as np

# Experiment A: magnitude and phase of 8 points on the unit circle
z8 = np.exp(1j * np.linspace(0, 2 * np.pi, 8, endpoint=False))
print('8 points on the unit circle (endpoint=False, avoids duplicating 0/2π)')
print(f'  magnitude: {np.abs(z8).round(4)}')  # all should be 1.0
print(f'  phase (radians): {np.angle(z8).round(4)}')  # 0, π/4, ..., π, then wraps around to -3π/4, -π/2, -π/4 (np.angle range is (-π, π])
assert np.allclose(np.abs(z8), 1.0), 'all magnitudes should be 1.0'
print('✅ magnitude is always 1, phases evenly spaced')

# Experiment B: negate imaginary part → phase negated, magnitude unchanged
z_pos = 3 + 4j
z_neg = 3 - 4j
print(f'\n3+4j: magnitude={abs(z_pos):.4f}, phase={np.angle(z_pos):.4f}')
print(f'3-4j: magnitude={abs(z_neg):.4f}, phase={np.angle(z_neg):.4f}')
assert abs(abs(z_pos) - abs(z_neg)) < 1e-9, 'magnitudes should be equal'
assert abs(np.angle(z_pos) + np.angle(z_neg)) < 1e-9, 'phases should be negatives of each other'
print('✅ negate imaginary part = negate phase, magnitude unchanged')


## 🖼️ Actually drawing the arrows: 8 points on the unit circle + "negative phase = measuring clockwise"

At the very start of the lesson we said "picture drawing an arrow on a blackboard", yet so far all you've seen is rows of numbers. Plotting `k=5 -> (-0.707, -0.707)` in your head is hard work — **this cell brings the blackboard to you**. matplotlib draws two plots at once:

- **Left plot**: the 8 points from the previous cell, drawn with arrows and angle labels. You'll see k=0..7 spread evenly counterclockwise, each arrow labeled with the actual `np.angle` reading — notice that from k=5 on, the labels suddenly turn negative;
- **Right plot**: explains that "sudden turn negative". Take k=5 (pointing 45° below the left horizontal): **the same arrow** measures +5π/4 counterclockwise (blue arc, most of the way around) and −3π/4 clockwise (green arc, less than halfway). `np.angle`'s rule is "always report the shortcut whose absolute value is at most π", so angles beyond π wrap to negative values — a negative phase isn't a new direction, just **the same direction measured turning the other way**.

(The plot labels use English/symbols to avoid font issues; CCW = counterclockwise, CW = clockwise.)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 5.4))

# ---- Left plot: 8 arrows on the unit circle, labeled with np.angle readings ----
ax = axes[0]
circle = np.exp(1j * np.linspace(0, 2 * np.pi, 200))
ax.plot(circle.real, circle.imag, ls='--', lw=1, color=plt.rcParams["grid.color"])
theta8 = np.linspace(0, 2 * np.pi, 8, endpoint=False)
z8 = np.exp(1j * theta8)
labels = ['0', '+π/4', '+π/2', '+3π/4', 'π', '−3π/4', '−π/2', '−π/4']  # actual np.angle output
for k, (zz, lab) in enumerate(zip(z8, labels)):
    ax.annotate('', xy=(zz.real, zz.imag), xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color='C0', lw=1.8))
    ax.annotate(f'k={k}\n{lab}', xy=(zz.real * 1.32, zz.imag * 1.32),
                ha='center', va='center', fontsize=9)
ax.axhline(0, color=plt.rcParams["axes.edgecolor"], lw=0.5); ax.axvline(0, color=plt.rcParams["axes.edgecolor"], lw=0.5)
ax.set_xlim(-1.7, 1.7); ax.set_ylim(-1.7, 1.7); ax.set_aspect('equal')
ax.set_xlabel('real (a)'); ax.set_ylabel('imag (b)')
ax.set_title('8 arrows on the unit circle\n(label = np.angle output)')

# ---- Right plot: two ways to measure the same arrow: +5π/4 (CCW) = −3π/4 (CW) ----
ax = axes[1]
ax.plot(circle.real, circle.imag, ls='--', lw=1, color=plt.rcParams["grid.color"])
zt = np.exp(1j * 5 * np.pi / 4)          # arrow pointing lower left (k=5 in the left plot)
ax.annotate('', xy=(zt.real, zt.imag), xytext=(0, 0),
            arrowprops=dict(arrowstyle='->', color='C3', lw=2.5))
arc_ccw = 0.40 * np.exp(1j * np.linspace(0, 5 * np.pi / 4, 120))    # measured CCW: +5π/4
ax.plot(arc_ccw.real, arc_ccw.imag, color='C0', lw=2)
arc_cw = 0.70 * np.exp(1j * np.linspace(0, -3 * np.pi / 4, 120))    # measured CW: −3π/4
ax.plot(arc_cw.real, arc_cw.imag, color='C2', lw=2)
ax.annotate('CCW: +5π/4', xy=(-0.15, 0.52), color='C0', fontsize=11, ha='center')
ax.annotate('CW: −3π/4', xy=(0.55, -0.72), color='C2', fontsize=11, ha='center')
ax.axhline(0, color=plt.rcParams["axes.edgecolor"], lw=0.5); ax.axvline(0, color=plt.rcParams["axes.edgecolor"], lw=0.5)
ax.set_xlim(-1.7, 1.7); ax.set_ylim(-1.7, 1.7); ax.set_aspect('equal')
ax.set_xlabel('real (a)'); ax.set_ylabel('imag (b)')
ax.set_title('Same arrow, two measurements:\n+5π/4 (CCW) = −3π/4 (CW)')

plt.tight_layout()
plt.show()
print('Left: k=0..7 spread counterclockwise; from k=5 the labels turn negative — that is np.angle switching to measuring clockwise.')
print('Right: there is only one red arrow; the blue arc (CCW +5π/4) and green arc (CW −3π/4) point to the same direction.')

## FFT output preview: every frequency bin is a complex number

No need to understand the derivation — just look at the output's data type and values, and build the intuition that "FFT output = an array of complex numbers".

In [ ]:
import numpy as np

# FFT the chord signal from L04 (Module 5 implements it from scratch; here we only look at the output shape)
sr, dur = 200, 1.0
t = np.arange(round(dur * sr)) / sr
chord = (np.sin(2*np.pi*20*t) +
         np.sin(2*np.pi*32*t) +
         np.sin(2*np.pi*40*t)) / 3  # three frequencies

X = np.fft.rfft(chord)  # FFT output: complex array
freqs = np.fft.rfftfreq(len(chord), 1/sr)

print(f'input:  {chord.shape}  float64')
print(f'output: {X.shape}  {X.dtype}  ← complex!')
print()

# find the three peak frequencies and print their complex values
magnitudes = np.abs(X)
# The next line chains three operations; unpacked (it's fine if you haven't seen these "lookup" tricks before):
#   Step 1 np.argsort(magnitudes): line up the *indices* by magnitude, smallest to largest (returns indices, not values)
#   Step 2 [-3:]: slice off the last 3 in line — the indices of the 3 largest magnitudes
#   Step 3 [::-1]: slice with step -1 walks backwards, flipping those 3 indices to largest-first
top3 = np.argsort(magnitudes)[-3:][::-1]
# Equivalent "beginner-readable version" (exactly the same result — verify it yourself):
#   order = np.argsort(magnitudes)   # indices, smallest to largest
#   top3 = list(reversed(order[len(order)-3:]))
print(f'  freq (Hz)  |  complex X[k]           |  magnitude (loudness) |  phase (rad)')
print('-' * 70)
for idx in sorted(top3):
    z = X[idx]
    mag, phase = abs(z), np.angle(z)
    print(f'  {freqs[idx]:>8.1f}  |  {z.real:+.2f} + {z.imag:+.2f}j  |  {mag:.2f}        |  {phase:.3f}')

print()
print('Magnitude ≈ 33 (amplitude 1/3 × samples/2 = 1/3 × 100 ≈ 33); the reason for this scaling (Parseval\'s theorem) is derived in L40 — just accept the number for now.')
print('Phase ≈ -π/2, because a sine wave is 0 at t=0 and cos leads it by π/2.')
print('  ↑ That sentence hides an unstated assumption: the FFT keeps its books with cos as the "zero-phase reference" —')
print('    why cos? That waits for the derivations in L06/L37 — accept it for now, like "magnitude ≈ 33" above;')
print('    but you needn\'t just wait to see that "the reference really is cos" — the next cell demonstrates it with a controlled experiment.')

### 🔬 Controlled experiment: what does the FFT's phase use as its "zero"?

The last line of the previous cell, "phase ≈ -π/2", may have given you a jolt: **I built the signal with sin — so why isn't the phase 0?** Who decided what the reference is?

Let's spell out the rule in one sentence: **the FFT keeps its books by writing every frequency component in the form `A·cos(2πft + φ)`** — the reference is **cos**, not sin. Why cos of all things? That waits for Euler's formula in L06 (cos is the real part of `e^{jθ}`, the natural "zero-phase face") and the DFT derivation in L37; for now, like "magnitude ≈ 33", **accept it as a conclusion**.

But "the reference is cos" itself needn't be taken on faith — we can test it on the spot. The logic goes like this:

1. If cos is the reference, then **feeding in a pure cosine wave**, the phase reading at the 20 Hz bin should be **0**;
2. L04 showed that sin lags cos by a quarter turn (90° = π/2), so **feeding in a pure sine wave**, the reading should be **−π/2**;
3. And one more "caught red-handed" case: feed in `cos(2πft + π/3)`, and the reading should be **+π/3** — the φ the FFT reads out is exactly the φ you wrote inside the cos parentheses.

Three predictions, all checked in the next cell. Hit all three, and "FFT phase = offset relative to cos" no longer falls out of the sky.

In [ ]:
import numpy as np

sr = 200
t = np.arange(200) / sr   # duration 1 s → frequency resolution 1 Hz → index 20 is exactly the 20 Hz bin

cases = [
    ('cos(2π·20·t)      ', np.cos(2*np.pi*20*t),           0.0),
    ('sin(2π·20·t)      ', np.sin(2*np.pi*20*t),           -np.pi/2),
    ('cos(2π·20·t + π/3)', np.cos(2*np.pi*20*t + np.pi/3), np.pi/3),
]
for name, sig, expected in cases:
    X = np.fft.rfft(sig)
    phase = np.angle(X[20])
    print(f'{name} → phase at the 20 Hz bin = {phase:+.4f} (expected {expected:+.4f})')
    assert np.isclose(phase, expected, atol=1e-9)

print()
print('✅ All three pieces of evidence are in: cos → phase 0 (it *is* the reference);')
print('   sin lags cos by 90° → phase -π/2; cos advanced by π/3 → phase +π/3.')
print('   Conclusion: the FFT keeps books as A·cos(2πft + φ); the phase φ it reads out is "the offset relative to cos".')

## 🎯 The payoff later

The **magnitude and phase** you read out of complex numbers by hand today will show up again and again in **L06 Euler's formula / the FFT and STFT in L35, L37–L44 / L70 Whisper** — there, every frequency bin is a complex number, its magnitude is "how loud this frequency is", and its phase is "how it lines up". Get it straight today, and you won't have to guess from scratch later.

## Wrap-up

You can now use your own `magnitude_phase(z)` to read the magnitude and phase out of any complex number;
after the assertions pass, you should be able to see that it is equivalent to `abs(z)` / `np.angle(z)` (calling those two directly was off-limits for the implementation).

**Key connections**:
- L04's `sinusoid(t, A=r, f=f0, phi=θ)` ↔ the `(magnitude=r, phase=θ)` of the FFT's complex output `X[k]`
- `magnitude_spectrogram = |FFT output|` (elementwise magnitude, implemented in L40)
- The phase spectrum is indispensable for waveform reconstruction (L45, inverse STFT)

**Next lesson L06**: Euler's formula `e^{jθ} = cos(θ) + j·sin(θ)` —
it links today's polar coordinates directly to last lesson's sine waves, and it's the shortest path to understanding the DFT matrix (L37).

## ✏️ Whiteboard challenge: complex numbers by hand (target: 8 minutes)

Cover the screen and answer on paper:

**Complex number**: z = 3 + 4j

**Question 1**: What is the magnitude of z? The formula is |z| = √(a² + b²).

**Question 2**: What is the phase of z in radians? Formula θ = arctan2(b, a) (work it by hand first, then run the answer-check cell).

**Question 3**: Rewrite z in polar form r·e^{iθ}, giving numerical values for r and θ.

**Question 4 (optional / L06 preview, not the main test of this lesson)**: What does `np.exp(1j * np.pi)` equal? (Euler's formula: e^{iπ} = cos(π) + i·sin(π) = ?)

> 📌 **How far does "by hand" need to go to pass?** For Question 2, getting on paper to **θ = arctan(4/3), and determining that it's in the first quadrant**, counts as correct — arctan(4/3) isn't a special angle like 30°/45°/60°, and nobody can get 0.9273 without a calculator; leave the four decimal places to the answer-check cell below. Want a rough sanity check of the magnitude on paper? In a 3-4-5 right triangle this angle is about **53°**, and 53 × π/180 ≈ **0.93 radians** — if that matches 0.9273, you haven't gone astray. Same for θ in Question 3: writing `arctan(4/3)` (or "about 0.93") is enough.
>
> 📌 **Question 4 is a preview of L06, not a test question for this lesson**: just plug into the formula given in the parentheses — cos(π) = −1, sin(π) = 0, which add up to −1 — and you're done. It tests "can you copy a formula and plug in values", not "do you understand why Euler's formula holds"; understanding is next lesson's job, and it's completely normal not to be able to answer "why" yet.

When you've finished the derivation, run the cell below to check your answers.

In [ ]:
# ✏️ Answer-check cell
import numpy as np

z = 3 + 4j

# Q1: magnitude
mag_theory = 5.0  # √(9+16) = √25 = 5
mag_code = abs(z)
assert np.isclose(mag_code, mag_theory, atol=1e-10)
print(f"Q1 ✅  |3+4j| = √(9+16) = 5  code value {mag_code:.4f}")

# Q2: phase
ph_theory = np.arctan2(4, 3)  # ≈ 0.9273 rad
ph_code = np.angle(z)
assert np.isclose(ph_code, ph_theory, atol=1e-10)
print(f"Q2 ✅  ∠(3+4j) = arctan2(4,3) ≈ {ph_theory:.4f} rad ({np.degrees(ph_theory):.2f}°)")

# Q3: polar form
r, theta = mag_code, ph_code
z_polar = r * np.exp(1j * theta)
assert np.isclose(z_polar.real, z.real, atol=1e-10)
assert np.isclose(z_polar.imag, z.imag, atol=1e-10)
print(f"Q3 ✅  z = {r:.1f}·e^{{i·{theta:.4f}}} = {z_polar.real:.1f} + {z_polar.imag:.1f}j (verified to match)")

# Q4: Euler's formula
z_euler = np.exp(1j * np.pi)
assert np.isclose(z_euler.real, -1.0, atol=1e-10)
assert np.isclose(z_euler.imag,  0.0, atol=1e-10)
print(f"Q4 ✅  e^{{iπ}} = cos(π) + i·sin(π) = -1 + 0j  code value {z_euler:.4f}")
print("\n🎉 Complex-number whiteboard challenge passed! Magnitude and phase are the two sides of a frequency's 'ID card'.")

In [ ]:
# ✏️ Self-review for this lesson
l05_review = {
    "magnitude_phase_implemented": None,  # magnitude_phase implemented and assertions pass? True/False
    "rectangular_polar_understood": None, # can convert rectangular ↔ polar coordinates? True/False
    # ↓ Criterion (this lesson only requires "can use it", not "understand why"): if you can use np.exp(1j*θ) as
    #   a shortcut for "input angle θ → output the matching point on the unit circle" and know it equals cos(θ)+sin(θ)j,
    #   you may fill in True. "Why e to an imaginary power goes in circles" is L06 material; not knowing it doesn't count as failing.
    "euler_formula_intuition":      None, # understand e^{iθ} is the point rotated by θ on the unit circle? True/False
    "whiteboard_passed":            None, # whiteboard challenge done on paper? True/False (Q2 counts as passed at arctan(4/3); see whiteboard cell notes)
}

unfilled = [k for k, v in l05_review.items() if v is None]
assert not unfilled, f'not filled in yet: {unfilled}'
weak = [k for k, v in l05_review.items() if v is False]
if weak:
    print(f'⚠️  needs more work: {weak}')
else:
    print('✅ L05 fully cleared! On to L06: Euler\'s formula e^{iθ}=cosθ+isinθ')

---

→ **Next lesson** · [L06 · Euler's formula e^{iθ}=cosθ+isinθ](L06_euler.en.ipynb)

> Next lesson covers **Euler's formula e^{iθ}=cosθ+isinθ**: the twiddle factor is the lifeblood of the FFT.